In [2]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
api_key = os.environ.get("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(
            model="gemini-3.6-flash",
            temperature=2.0,
            google_api_key=api_key)


In [11]:
from langchain_core.prompts import ChatPromptTemplate

# A clean layout that Gemini understands perfectly
mat_cot_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful math tutor. Break down the problem step-by-step using a Chain of Thought approach."),
    ("user", "Solve equation {question}")
])

cot_chain = mat_cot_prompt | llm | StrOutputParser()
print(cot_chain.invoke({"question": "Solve equation 2*x+5=15"}))


C:\Users\Shivachetan Ulavi\Anaconda3\envs\langchain-book\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Hello! I would be happy to help you solve this step-by-step. 

Our goal is to get the variable **$x$** all by itself on one side of the equals sign.

Here is the original equation:
$$2x + 5 = 15$$

---

### **Step 1: Isolate the term with $x$**
Before we can get $x$ by itself, we need to get $2x$ by itself. To do this, we need to undo the addition of $5$. 

The opposite of adding $5$ is subtracting $5$. Remember, whatever we do to one side of the equation, we must do to the other side to keep it balanced.

* Subtract $5$ from both sides:
  $$2x + 5 - 5 = 15 - 5$$

* Simplify both sides:
  $$2x = 10$$

---

### **Step 2: Solve for $x$**
Now we have $2x = 10$. 
$2x$ means "$2$ multiplied by $x$". To undo multiplication, we perform the opposite operation, which is division.

* Divide both sides by $2$:
  $$\frac{2x}{2} = \frac{10}{2}$$

* Simplify both sides:
  $$x = 5$$

---

### **Step 3: Check your answer**
It's always a good idea to plug our answer back into the original equation to m

## Getting a concise answer

In [13]:
from operator import itemgetter

In [14]:
from langchain_core.prompts import PromptTemplate

In [16]:
parse_prompt_template = (
"Given the initial question and a full answer, "
"extract the concise answer. Do not assume anything and "
"only use a provided full answer.\n\nQUESTION:\n{question}\n"
"FULL ANSWER:\n{full_answer}\n\nCONCISE ANSWER:\n"
)

parse_prompt = PromptTemplate.from_template(parse_prompt_template)

In [18]:
final_chain = (
    {"full_answer":itemgetter("question") | cot_chain,
     "question":itemgetter("question"),}
    |parse_prompt
    |llm
    |StrOutputParser()
)

In [19]:
print(final_chain.invoke({"question" : "Solve Equation 2*x + 5 = 15"}))

C:\Users\Shivachetan Ulavi\Anaconda3\envs\langchain-book\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\Shivachetan Ulavi\Anaconda3\envs\langchain-book\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


x = 5


### Checking for Self Consistency

In [20]:
generations = []
for _ in range(20):
    generations.append(final_chain.invoke({"question": "Solve equation 2*x**2 - 96*x + 1152"}, temperature = 2.0).strip())

C:\Users\Shivachetan Ulavi\Anaconda3\envs\langchain-book\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\Shivachetan Ulavi\Anaconda3\envs\langchain-book\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\Shivachetan Ulavi\Anaconda3\envs\langchain-book\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\Shivachetan Ulavi\Anaconda3\envs\langchain-book\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.6-flas

GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 16.086419895s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

In [21]:
from collections import Counter
print(Counter(generations).most_common(1)[0][0])

x = 24


In [22]:
len(generations)

7

In [23]:
generations

['x = 24', 'x = 24', 'x = 24', 'x = 24', 'x = 24', 'x = 24', 'x = 24']

## Using local deepseek model

In [25]:
from langchain_ollama import ChatOllama
#from langchain_core.prompts import PromptTemplate

print("🔄 Connecting to local Ollama instance...")
llm2 = ChatOllama(
        model="deepseek-r1:1.5b", # You can also switch this to "deepseek-r1:1.5b" to test that one!
        temperature=2.0)

🔄 Connecting to local Ollama instance...


In [26]:
test = "hello, how are you?"
llm2.invoke(test)

AIMessage(content="\n\nHello! I'm just a virtual assistant, so I don't have feelings, but I'm here and ready to help you with whatever you need. How are you doing? 😊", additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-08-29T08:57:56.6324003Z', 'done': True, 'done_reason': 'stop', 'total_duration': 17121542000, 'load_duration': 14635416900, 'prompt_eval_count': 9, 'prompt_eval_duration': 108251000, 'eval_count': 42, 'eval_duration': 2357769000, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--01a04cbd-25aa-71f1-b743-b9a55671646d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 42, 'total_tokens': 51})

In [32]:
# A clean layout that Gemini understands perfectly
mat_cot_prompt2 = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful math tutor. Break down the problem step-by-step using a Chain of Thought approach."),
    ("user", "Solve equation {question}")
])

cot_chain2 = mat_cot_prompt2 | llm2 | StrOutputParser()
print(cot_chain2.invoke({"question": "Solve equation 2*x+5=15"}))




Sure! Let's solve the equation step by step.

**Problem:**  
Solve the equation \(2x + 5 = 15\).

**Solution:**

1. **Isolate the term with the variable (x):**
   - First, subtract 5 from both sides of the equation to get rid of the constant term on the left side.
   \[
   2x + 5 - 5 = 15 - 5
   \]
   Simplifying both sides:
   \[
   2x = 10
   \]

2. **Solve for x:**
   - Now, divide both sides of the equation by 2 to isolate x.
   \[
   \frac{2x}{2} = \frac{10}{2}
   \]
   Simplifying both sides:
   \[
   x = 5
   \]

3. **Verification:**
   - To ensure the solution is correct, substitute \(x = 5\) back into the original equation.
   \[
   2(5) + 5 = 15
   \]
   Simplify:
   \[
   10 + 5 = 15
   \]
   Which checks out.

**Conclusion:**  
The solution to the equation \(2x + 5 = 15\) is \(x = 5\).


In [33]:
final_chain2 = (
    {"full_answer":itemgetter("question") | cot_chain2,
     "question":itemgetter("question"),}
    |parse_prompt
    |llm2
    |StrOutputParser()
)

In [34]:
print(final_chain2.invoke({"question" : "Solve Equation 2*x + 5 = 15"}))



The solution to the equation is x = 5.


In [35]:
generations2 = []
for _ in range(20):
    generations2.append(final_chain2.invoke({"question": "Solve equation 2*x**2 - 96*x + 1152"}, temperature = 2.0).strip())

In [36]:
from collections import Counter
print(Counter(generations2).most_common(1)[0][0])

The quadratic equation \( 2x^2 - 96x + 1152 = 0 \) can be simplified and solved as follows:

1. Factor out the GCF, which is 2: \( 2(x^2 - 48x + 576) = 0 \).
2. Recognize the perfect square trinomial: \( (x - 24)^2 = 0 \).
3. Solve for \( x \): \( x = 24 \).

**Final Solution:**

\[
x = 24
\]


In [38]:
generations2[-1]

'The equation \\(2x^2 - 96x + 1152 = 0\\) simplifies to \\((x - 24)^2 = 0\\), which means the solution is \\(x = 24\\).\n\n\\[\n\\boxed{24}\n\\]'

In [39]:
len(generations2)

20